# PCA from Scratch — Building Intuition with Iris Dataset

**Goal**: You know eigenvectors & eigenvalues. Now let's see WHY they matter for data.

We'll go step by step:
1. Look at the raw data (4D — we can't see it all)
2. Compute the covariance matrix (how features relate)
3. Get eigenvectors/eigenvalues (directions of spread)
4. Project data onto top eigenvectors (dimensionality reduction)
5. Compare with sklearn's PCA (same thing, one line)

---

In [45]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import pandas as pd

# Load Iris dataset
iris = load_iris()
X = iris.data                    # shape: (150, 4)
y = iris.target                  # 0, 1, 2 for three species
feature_names = iris.feature_names
target_names = iris.target_names

df = pd.DataFrame(X, columns=feature_names)
df['species'] = [target_names[i] for i in y]

print(f"Shape: {X.shape} — 150 samples, 4 features")
print(f"Features: {feature_names}")
print(f"Species: {list(target_names)}")
df.head(10)

Shape: (150, 4) — 150 samples, 4 features
Features: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Species: ['setosa', 'versicolor', 'virginica']


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa
5,5.4,3.9,1.7,0.4,setosa
6,4.6,3.4,1.4,0.3,setosa
7,5.0,3.4,1.5,0.2,setosa
8,4.4,2.9,1.4,0.2,setosa
9,4.9,3.1,1.5,0.1,setosa


## Step 1: The Problem — We Can't See 4D

Let's try looking at pairs of features. We get *some* separation, but we're only seeing 2 out of 4 dimensions at a time. We might be missing the best angle.

In [46]:
# Scatter matrix — every pair of features
fig = px.scatter_matrix(
    df, 
    dimensions=feature_names, 
    color='species',
    title='Iris: All Pairwise Feature Plots (we only see 2D slices of 4D data)',
    opacity=0.7,
    height=700, width=800
)
fig.update_traces(diagonal_visible=False, marker=dict(size=4))
fig.show()

**Notice**: Some pairs separate the species better than others. But what if the BEST view isn't along any single feature axis? What if it's along a *diagonal* direction through 4D space?

That's exactly what PCA finds.

---

## Step 2: Standardize the Data

Before PCA, we center (and often scale) the data. Why?
- **Centering** (subtract mean): PCA finds directions from the origin. If data isn't centered, the first PC just points at the mean — useless.
- **Scaling** (divide by std): If one feature is in cm and another in km, the km feature dominates just because of units. Scaling puts them on equal footing.

For Iris, all features are in cm so scaling is less critical, but it's good practice.

In [47]:
# Standardize: zero mean, unit variance
scaler = StandardScaler()
X_std = scaler.fit_transform(X)

print("Before standardization:")
print(f"  Means:  {X.mean(axis=0).round(2)}")
print(f"  Stds:   {X.std(axis=0).round(2)}")
print()
print("After standardization:")
print(f"  Means:  {X_std.mean(axis=0).round(6)}  (≈ 0)")
print(f"  Stds:   {X_std.std(axis=0).round(6)}  (≈ 1)")

Before standardization:
  Means:  [5.84 3.06 3.76 1.2 ]
  Stds:   [0.83 0.43 1.76 0.76]

After standardization:
  Means:  [-0. -0. -0. -0.]  (≈ 0)
  Stds:   [1. 1. 1. 1.]  (≈ 1)


In [48]:
print("variance of each feature (before standardization):", X.var(axis=0).round(2))
print("variance of each feature (after standardization):", X_std.var(axis=0).round(6))

variance of each feature (before standardization): [0.68 0.19 3.1  0.58]
variance of each feature (after standardization): [1. 1. 1. 1.]


In [49]:
'''
standard deviation (std) is the square root of variance, so std^2 = variance.
Before standardization, the variances of the features are not all the same, which means they are on different scales. After standardization, the variances are all approximately 1, which means
scaling happens like this :
X_std = (X - mean) / std
so the variance of X_std is:
Var(X_std) = Var((X - mean) / std) = Var(X) / (std^2) = Var(X) / Var(X) = 1
'''

'\nstandard deviation (std) is the square root of variance, so std^2 = variance.\nBefore standardization, the variances of the features are not all the same, which means they are on different scales. After standardization, the variances are all approximately 1, which means\nscaling happens like this :\nX_std = (X - mean) / std\nso the variance of X_std is:\nVar(X_std) = Var((X - mean) / std) = Var(X) / (std^2) = Var(X) / Var(X) = 1\n'

---

## Step 3: The Covariance Matrix — How Features Move Together

The covariance matrix tells you: when feature A goes up, does feature B go up too (positive covariance), go down (negative), or not care (near zero)?

It's a 4×4 matrix (since we have 4 features). The diagonal = variance of each feature. Off-diagonal = covariance between pairs.

In [50]:
# Covariance matrix (on standardized data)
# np.cov expects features as rows, so we transpose; rowvar=False also works
cov_matrix = np.cov(X_std, rowvar=False)

print("Covariance Matrix (4×4):")
print(np.round(cov_matrix, 3))
print()

# Visualize it as a heatmap
fig = px.imshow(
    cov_matrix,
    x=feature_names, y=feature_names,
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='Covariance Matrix Heatmap',
    text_auto='.2f',
    height=500, width=550
)
fig.show()

Covariance Matrix (4×4):
[[ 1.007 -0.118  0.878  0.823]
 [-0.118  1.007 -0.431 -0.369]
 [ 0.878 -0.431  1.007  0.969]
 [ 0.823 -0.369  0.969  1.007]]



**Read this heatmap**: 
- Petal length & petal width have HIGH positive covariance (~0.96). They move together.
- Sepal length & petal length also correlate strongly (~0.87).
- Sepal width is kind of independent from the others (low/negative covariance).

This tells us: there's redundancy! Some features carry overlapping information. PCA will find the "true" underlying directions.

---

## Step 4: Eigen-decomposition — THIS is where your knowledge kicks in

Now we find the eigenvectors and eigenvalues of this covariance matrix.

- **Eigenvectors** = the principal component directions (new axes)
- **Eigenvalues** = how much variance is along each direction

In [51]:
# Eigen-decomposition of the covariance matrix
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
print("Eigenvalues:", eigenvalues)
print("Eigenvectors (columns):")
print(eigenvectors)
# eigh returns in ascending order — we want descending (largest first)
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

print("Eigenvalues (amount of variance in each direction):")
for i, ev in enumerate(eigenvalues):
    pct = 100 * ev / eigenvalues.sum()
    print(f"  PC{i+1}: {ev:.4f}  ({pct:.1f}% of total variance)")

print(f"\nTotal variance: {eigenvalues.sum():.4f} (should be ~4, since 4 standardized features)")
print(f"\nPC1 + PC2 together: {100 * eigenvalues[:2].sum() / eigenvalues.sum():.1f}% of variance")
print("→ We can go from 4D to 2D and keep most of the information!")

Eigenvalues: [0.02085386 0.14774182 0.9201649  2.93808505]
Eigenvectors (columns):
[[ 0.26128628  0.71956635  0.37741762 -0.52106591]
 [-0.12350962 -0.24438178  0.92329566  0.26934744]
 [-0.80144925 -0.14212637  0.02449161 -0.5804131 ]
 [ 0.52359713 -0.63427274  0.06694199 -0.56485654]]
Eigenvalues (amount of variance in each direction):
  PC1: 2.9381  (73.0% of total variance)
  PC2: 0.9202  (22.9% of total variance)
  PC3: 0.1477  (3.7% of total variance)
  PC4: 0.0209  (0.5% of total variance)

Total variance: 4.0268 (should be ~4, since 4 standardized features)

PC1 + PC2 together: 95.8% of variance
→ We can go from 4D to 2D and keep most of the information!


In [52]:
# What do the eigenvectors look like? Each is a 4D direction.
print("Eigenvectors (each column = one principal component direction):")
print("These tell you the RECIPE: how much of each original feature goes into each PC.\n")

for i in range(4):
    print(f"PC{i+1} = ", end="")
    parts = []
    for j, fname in enumerate(feature_names):
        w = eigenvectors[j, i]
        parts.append(f"{w:+.3f}×{fname}")
    print(" + ".join(parts).replace("+ -", "- "))

print("\n→ PC1 is roughly: big petal length + big petal width + big sepal length - some sepal width")
print("→ It's capturing overall 'flower size' (except sepal width goes the other way)")

Eigenvectors (each column = one principal component direction):
These tell you the RECIPE: how much of each original feature goes into each PC.

PC1 = -0.521×sepal length (cm) + +0.269×sepal width (cm) - 0.580×petal length (cm) - 0.565×petal width (cm)
PC2 = +0.377×sepal length (cm) + +0.923×sepal width (cm) + +0.024×petal length (cm) + +0.067×petal width (cm)
PC3 = +0.720×sepal length (cm) - 0.244×sepal width (cm) - 0.142×petal length (cm) - 0.634×petal width (cm)
PC4 = +0.261×sepal length (cm) - 0.124×sepal width (cm) - 0.801×petal length (cm) + +0.524×petal width (cm)

→ PC1 is roughly: big petal length + big petal width + big sepal length - some sepal width
→ It's capturing overall 'flower size' (except sepal width goes the other way)


In [53]:
# Scree plot — how much variance each PC captures
explained_var = eigenvalues / eigenvalues.sum() * 100
cumulative_var = np.cumsum(explained_var)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=[f'PC{i+1}' for i in range(4)],
    y=explained_var,
    name='Individual',
    marker_color='steelblue',
    text=[f'{v:.1f}%' for v in explained_var],
    textposition='outside'
))
fig.add_trace(go.Scatter(
    x=[f'PC{i+1}' for i in range(4)],
    y=cumulative_var,
    name='Cumulative',
    marker_color='firebrick',
    mode='lines+markers+text',
    text=[f'{v:.1f}%' for v in cumulative_var],
    textposition='top center'
))
fig.update_layout(
    title='Scree Plot: How Much Variance Does Each PC Capture?',
    yaxis_title='Explained Variance (%)',
    height=450, width=650,
    legend=dict(x=0.7, y=0.5)
)
fig.show()

**Key takeaway from the scree plot**:
- PC1 alone captures ~73% of the variance
- PC1 + PC2 together capture ~96%
- PC3 and PC4 add very little — they're mostly noise

So we can safely drop from 4D → 2D and lose almost nothing. That's the magic.

---

## Step 5: Project the Data — The Actual Dimensionality Reduction

In [54]:
# Project: multiply data by the top 2 eigenvectors
# X_std is (150, 4), eigenvectors[:, :2] is (4, 2)
# Result: (150, 2) — each sample now has 2 coordinates instead of 4

W = eigenvectors[:, :2]  # Take first 2 principal components
X_pca = X_std @ W        # Matrix multiplication = projection

print(W)
print(X_std[:5])

print(f"Original shape: {X_std.shape}  →  Projected shape: {X_pca.shape}")
print(f"\nFirst 5 samples in PCA space:")
print(X_pca[:5].round(3))

[[-0.52106591  0.37741762]
 [ 0.26934744  0.92329566]
 [-0.5804131   0.02449161]
 [-0.56485654  0.06694199]]
[[-0.90068117  1.01900435 -1.34022653 -1.3154443 ]
 [-1.14301691 -0.13197948 -1.34022653 -1.3154443 ]
 [-1.38535265  0.32841405 -1.39706395 -1.3154443 ]
 [-1.50652052  0.09821729 -1.2833891  -1.3154443 ]
 [-1.02184904  1.24920112 -1.34022653 -1.3154443 ]]
Original shape: (150, 4)  →  Projected shape: (150, 2)

First 5 samples in PCA space:
[[ 2.265  0.48 ]
 [ 2.081 -0.674]
 [ 2.364 -0.342]
 [ 2.299 -0.597]
 [ 2.39   0.647]]


In [55]:
# THE BIG REVEAL: plot the 2D projection
pca_df = pd.DataFrame({
    'PC1': X_pca[:, 0],
    'PC2': X_pca[:, 1],
    'species': [target_names[i] for i in y]
})

fig = px.scatter(
    pca_df, x='PC1', y='PC2', color='species',
    title=f'PCA Projection (2D captures {cumulative_var[1]:.1f}% of 4D variance)',
    height=550, width=700,
    opacity=0.8
)
fig.update_traces(marker=dict(size=8, line=dict(width=1, color='white')))
fig.update_layout(
    xaxis_title=f'PC1 ({explained_var[0]:.1f}% variance)',
    yaxis_title=f'PC2 ({explained_var[1]:.1f}% variance)'
)
fig.show()

**Look at that!** Three species clearly visible in just 2D. Setosa is completely separated. Versicolor and Virginica overlap a bit (that's real — they're genuinely similar species).

This is a BETTER view than any single pair of original features could give us, because PC1 and PC2 are *combinations* of all 4 features, oriented to show maximum spread.

---

## Step 6: What Do the PCs Actually Mean? (Loading Vectors)

In [56]:
# Biplot: shows both data points AND feature contribution arrows
# This tells you WHAT each PC is measuring in terms of original features

fig = px.scatter(
    pca_df, x='PC1', y='PC2', color='species',
    title='Biplot: Data + Feature Loadings (arrows show what each PC means)',
    height=600, width=750,
    opacity=0.6
)
fig.update_traces(marker=dict(size=7, line=dict(width=0.5, color='white')))

# Add arrows for each original feature
scale = 3  # scale arrows for visibility
colors_arrow = ['#e74c3c', '#2ecc71', '#3498db', '#f39c12']
for i, fname in enumerate(feature_names):
    fig.add_annotation(
        ax=0, ay=0,
        x=eigenvectors[i, 0] * scale,
        y=eigenvectors[i, 1] * scale,
        xref='x', yref='y', axref='x', ayref='y',
        showarrow=True,
        arrowhead=3, arrowsize=1.5, arrowwidth=2.5,
        arrowcolor=colors_arrow[i]
    )
    fig.add_annotation(
        x=eigenvectors[i, 0] * scale * 1.15,
        y=eigenvectors[i, 1] * scale * 1.15,
        text=f"<b>{fname}</b>",
        showarrow=False,
        font=dict(size=11, color=colors_arrow[i])
    )

fig.update_layout(
    xaxis_title=f'PC1 ({explained_var[0]:.1f}%)',
    yaxis_title=f'PC2 ({explained_var[1]:.1f}%)'
)
fig.show()

**Reading the biplot**:
- Arrows pointing in similar directions = those features are correlated
- Petal length, petal width, sepal length all point right → PC1 ≈ "overall flower size"
- Sepal width points up → PC2 captures sepal width variation
- Arrow length = how much that feature contributes to the PC

---

## Step 7: Verify — sklearn PCA gives the same result

In [57]:
# One line with sklearn — does the same thing we just did manually
pca_sklearn = PCA(n_components=2)
X_pca_sklearn = pca_sklearn.fit_transform(X_std)

# Note: signs might be flipped (eigenvectors can point either way — both valid)
print("Our manual eigenvalues:  ", eigenvalues[:2].round(4))
print("sklearn eigenvalues:     ", pca_sklearn.explained_variance_.round(4))
print()
print("Our variance ratios:     ", (eigenvalues[:2] / eigenvalues.sum()).round(4))
print("sklearn variance ratios: ", pca_sklearn.explained_variance_ratio_.round(4))
print()

# Check if projections match (up to sign flip)
diff = np.min([
    np.abs(X_pca - X_pca_sklearn).max(),
    np.abs(X_pca + X_pca_sklearn).max(),  # sign-flipped version
    np.abs(X_pca - X_pca_sklearn * np.array([[-1, 1]])).max(),
    np.abs(X_pca - X_pca_sklearn * np.array([[1, -1]])).max()
])
print(f"Max difference (accounting for sign flips): {diff:.10f}")
print("→ Same thing! sklearn just wraps what we did by hand.")

Our manual eigenvalues:   [2.9381 0.9202]
sklearn eigenvalues:      [2.9381 0.9202]

Our variance ratios:      [0.7296 0.2285]
sklearn variance ratios:  [0.7296 0.2285]

Max difference (accounting for sign flips): 0.0000000000
→ Same thing! sklearn just wraps what we did by hand.


---

## Step 8: Reconstruction — What Do We Lose?

If we project to 2D and then project BACK to 4D, how close are we to the original? This shows what information PCA preserves vs. throws away.

In [58]:
# Reconstruct: go from 2D back to 4D
X_reconstructed = X_pca @ W.T  # (150,2) × (2,4) = (150,4)

# Compare original vs reconstructed for the first sample
print("Sample 0 comparison (standardized space):")
print(f"  Original:      {X_std[0].round(3)}")
print(f"  Reconstructed: {X_reconstructed[0].round(3)}")
print(f"  Difference:    {(X_std[0] - X_reconstructed[0]).round(3)}")
print()

# Overall reconstruction error
mse = np.mean((X_std - X_reconstructed) ** 2)
print(f"Mean Squared Error across all samples: {mse:.4f}")
print(f"This represents the {100 - cumulative_var[1]:.1f}% variance we dropped (PC3 + PC4)")

Sample 0 comparison (standardized space):
  Original:      [-0.901  1.019 -1.34  -1.315]
  Reconstructed: [-0.999  1.053 -1.303 -1.247]
  Difference:    [ 0.098 -0.034 -0.038 -0.068]

Mean Squared Error across all samples: 0.0419
This represents the 4.2% variance we dropped (PC3 + PC4)


In [59]:
# Visual: original vs reconstructed for each feature
fig = make_subplots(rows=2, cols=2, subplot_titles=feature_names)

for i, fname in enumerate(feature_names):
    row, col = (i // 2) + 1, (i % 2) + 1
    fig.add_trace(go.Scatter(
        x=X_std[:, i], y=X_reconstructed[:, i],
        mode='markers', marker=dict(size=4, opacity=0.5,
        color=[['steelblue','darkorange','forestgreen'][t] for t in y]),
        showlegend=False
    ), row=row, col=col)
    # Perfect reconstruction line
    rng = [min(X_std[:, i].min(), X_reconstructed[:, i].min()),
           max(X_std[:, i].max(), X_reconstructed[:, i].max())]
    fig.add_trace(go.Scatter(
        x=rng, y=rng, mode='lines',
        line=dict(color='red', dash='dash', width=1),
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    title='Original vs Reconstructed (closer to red line = better preserved)',
    height=600, width=700
)
fig.show()

---

## Step 9: 3D View — Because Plotly Can

Let's use 3 PCs for a 3D view. Drag to rotate!

In [60]:
# 3D PCA projection
X_pca_3d = X_std @ eigenvectors[:, :3]

pca3d_df = pd.DataFrame({
    'PC1': X_pca_3d[:, 0],
    'PC2': X_pca_3d[:, 1],
    'PC3': X_pca_3d[:, 2],
    'species': [target_names[i] for i in y]
})

fig = px.scatter_3d(
    pca3d_df, x='PC1', y='PC2', z='PC3', color='species',
    title=f'3D PCA ({cumulative_var[2]:.1f}% variance captured) — drag to rotate!',
    height=650, width=800,
    opacity=0.8
)
fig.update_traces(marker=dict(size=5))
fig.show()

---

## Summary: The PCA Recipe

Here's what we did, and this is ALL PCA is:

```
1. Standardize data          (zero mean, unit variance)
2. Compute covariance matrix (how features relate)
3. Eigendecomposition        (find directions + their importance)
4. Sort by eigenvalue        (most important first)
5. Pick top k eigenvectors   (how many dimensions to keep)
6. Project: X_new = X @ W   (multiply data by chosen eigenvectors)
```

That's it. sklearn's `PCA()` does all 6 steps in one call.

---

## What's Next?

Now that you understand PCA, the natural progression in EECE5644 is:

1. **LDA (Linear Discriminant Analysis)** — Like PCA but supervised. Instead of maximizing variance, it maximizes class separation. Uses between-class and within-class scatter matrices. You'll see this soon.

2. **Probability & Bayes Decision Theory** — The backbone of the class. How to make optimal decisions under uncertainty.

3. **Gaussian classifiers, MAP, MLE** — Modeling data as Gaussian distributions and using them to classify.

4. **Clustering (K-means, GMM, EM)** — Unsupervised grouping of data.

5. **Nonlinear methods** — Kernel PCA, neural networks, SVMs.

One step at a time. Get comfortable with PCA first, then we'll tackle whatever's next in Brady's syllabus.